In [1]:
# %% [markdown]
# # Notebook 05 — Test End-to-End | Pipeline Complet IA + Blockchain
#
# **Mémoire** : L'IA et la Blockchain au service de la lutte contre
#               la piraterie musicale
# **Auteur**  : ADJINDA Adékin Olatobi Algis
# **Directeur** : Pr. Eugène EZIN
#
# ---
#
# ## Objectif
#
# Valider les deux flux opérationnels du système dans leur intégralité,
# sans passer par l'API FastAPI, en appelant directement les classes
# Python des couches IA, IPFS et Blockchain.
#
# ## Flux couverts
#
# | Flux | Étapes | Statut |
# |------|--------|--------|
# | Flux 1 — Piraterie | Enregistrement → Détection → Preuve → DMCA | Opérationnel (étape 4 simulée) |
# | Flux 2 — Redevances | Sélection œuvre → simulateRoyaltyPayment() | Simulé on-chain |
#
# ## Prérequis
#
# ```bash
# # Terminal 1 — Ganache
# npx ganache --port 7545 --chainId 1337
#
# # Terminal 2 — IPFS
# ipfs daemon
#
# # Terminal 3 — Déploiement contrat (si pas déjà fait)
# cd blockchain && npx hardhat run scripts/deploy.js --network ganache
# ```
#
# ## Métriques capturées
#
# - Précision de détection (score de similarité)
# - Absence de faux positifs
# - Latence de chaque étape (ms)
# - Coût gas de chaque transaction
# - Intégrité IPFS des preuves générées

# %% [markdown]
# ---
# ## 0. Initialisation et vérification des prérequis

# %%
import sys
import os
import time
import json
import uuid
import hashlib
import io
import tempfile
import librosa
import warnings
import requests as _req
from datetime  import datetime
from pathlib   import Path

import numpy    as np
import soundfile as sf

warnings.filterwarnings("ignore")

# ── Chemins ───────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.path.abspath(".."))
sys.path.insert(0, str(PROJECT_ROOT))

from web3 import Web3

from backend.core.fingerprint_engine import FingerprintEngine
from backend.core.ipfs_storage       import IPFSStorage
from backend.core.blockchain_manager import BlockchainManager
from backend.core.piracy_detector    import PiracyDetector
from backend.db.fingerprint_index    import FingerprintIndex
from backend.models.match_result     import MatchResult
from backend.models.proof            import Proof
from backend.models.takedown_request import TakedownRequest, TakedownStatus
from backend.config                  import settings

# ── Couleurs terminal ─────────────────────────────────────────────────────────
GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
BLUE   = "\033[94m"
BOLD   = "\033[1m"
RESET  = "\033[0m"
OK     = f"{GREEN}✅{RESET}"
FAIL   = f"{RED}❌{RESET}"
WARN   = f"{YELLOW}⚠️ {RESET}"
INFO   = f"{BLUE}ℹ️ {RESET}"

def header(title: str, color: str = BLUE):
    width = 62
    print(f"\n{color}{BOLD}{'═' * width}{RESET}")
    print(f"{color}{BOLD}  {title}{RESET}")
    print(f"{color}{BOLD}{'═' * width}{RESET}")

def section(title: str):
    print(f"\n{BOLD}── {title} {'─' * (56 - len(title))}{RESET}")

def result_row(label: str, value: str, status: str = ""):
    pad = 35
    print(f"  {label:<{pad}} {value}  {status}")

# %%
header("VÉRIFICATION DES PRÉREQUIS", BLUE)

checks = {}

# 1. Fichiers de configuration
section("Fichiers")
for label, path in [
    ("deployment.json", settings.deployment_json),
    ("fingerprints.json (optionnel)", settings.fingerprint_db),
]:
    exists = Path(path).exists()
    checks[label] = exists
    result_row(label, str(path)[-40:], OK if exists else WARN)

# 2. Services externes
section("Services")
# IPFS
try:
    r = _req.post(f"{settings.ipfs_api_url}/api/v0/id", timeout=3)
    ipfs_ok = r.status_code == 200
except Exception:
    ipfs_ok = False
checks["IPFS"] = ipfs_ok
result_row("IPFS daemon", settings.ipfs_api_url, OK if ipfs_ok else FAIL)

# Ganache
try:
    w3_test = Web3(Web3.HTTPProvider(settings.ganache_url))
    ganache_ok = w3_test.is_connected()
except Exception:
    ganache_ok = False
checks["Ganache"] = ganache_ok
result_row("Ganache", settings.ganache_url, OK if ganache_ok else FAIL)

# Résumé
all_critical = checks.get("deployment.json", False) and ganache_ok and ipfs_ok
print(f"\n  {'─' * 50}")
print(f"  Prérequis critiques : "
      f"{'✅ TOUS VÉRIFIÉS' if all_critical else '❌ VÉRIFIER CI-DESSUS'}")

if not all_critical:
    print(f"\n{RED}  Arrêt — vérifier les prérequis avant de continuer.{RESET}")
    print(f"  1. Ganache  : npx ganache --port 7545 --chainId 1337")
    print(f"  2. IPFS     : ipfs daemon")
    print(f"  3. Contrat  : cd blockchain && npx hardhat run scripts/deploy.js --network ganache")

# %% [markdown]
# ---
# ## 1. Initialisation des composants

# %%
header("INITIALISATION DES COMPOSANTS", BLUE)

# ── FingerprintEngine ─────────────────────────────────────────────────────────
section("FingerprintEngine (GraFPrint + Librosa)")
t0     = time.perf_counter()
engine = FingerprintEngine()
engine.load_grafprint_model()
t_load = (time.perf_counter() - t0) * 1000

model_status = "GraFPrint GNN" if engine.model else "Fallback Librosa"
result_row("Mode",              model_status, OK)
result_row("Device",            str(engine.device), INFO)
result_row("Temps chargement",  f"{t_load:.1f}ms", INFO)

# ── IPFSStorage ───────────────────────────────────────────────────────────────
section("IPFSStorage")
ipfs = IPFSStorage()
result_row("API URL",    ipfs.api_url, INFO)
result_row("Gateway",    ipfs.gateway_url, INFO)
result_row("Disponible", "OUI" if ipfs.is_available() else "NON",
           OK if ipfs.is_available() else FAIL)

# ── BlockchainManager ─────────────────────────────────────────────────────────
section("BlockchainManager (Web3.py)")
t0 = time.perf_counter()
blockchain = BlockchainManager()
blockchain.connect_web3()
t_connect = (time.perf_counter() - t0) * 1000

w3       = Web3(Web3.HTTPProvider(settings.ganache_url))
ACCOUNTS = w3.eth.accounts
DEPLOYER = ACCOUNTS[0]
ARTIST   = ACCOUNTS[1]
PRODUCER = ACCOUNTS[2]
LABEL    = ACCOUNTS[3]

with open(settings.deployment_json) as f:
    dep = json.load(f)

result_row("Connecté",        "OUI" if blockchain.is_connected() else "NON",
           OK if blockchain.is_connected() else FAIL)
result_row("Contrat",         dep["address"][:22] + "...", INFO)
result_row("Block courant",   str(w3.eth.block_number), INFO)
result_row("Comptes Ganache", str(len(ACCOUNTS)), INFO)
result_row("Temps connexion", f"{t_connect:.1f}ms", INFO)

# Soldes
section("Comptes de test")
for label, addr in [("Déployeur", DEPLOYER), ("Artiste", ARTIST),
                    ("Producteur", PRODUCER), ("Label", LABEL)]:
    bal = w3.from_wei(w3.eth.get_balance(addr), 'ether')
    result_row(f"{label}", f"{addr[:20]}...  {bal:.4f} ETH", OK)

# ── FingerprintIndex ──────────────────────────────────────────────────────────
section("FingerprintIndex")
index = FingerprintIndex(settings.fingerprint_db)
result_row("Empreintes préexistantes", str(index.count()), INFO)

# ── PiracyDetector ────────────────────────────────────────────────────────────
section("PiracyDetector")
detector = PiracyDetector(
    fingerprinter = engine,
    index         = index,
    ipfs          = ipfs,
    blockchain    = blockchain,
    threshold     = 0.85
)
result_row("Seuil de détection", str(detector._threshold), INFO)
result_row("Statut", "Prêt", OK)

print(f"\n{GREEN}{BOLD}  Tous les composants initialisés avec succès{RESET}")


══════════════════════════════════════════════════════════════
  VÉRIFICATION DES PRÉREQUIS
══════════════════════════════════════════════════════════════

── Fichiers ────────────────────────────────────────────────
  deployment.json                     ic-antipiracy\backend\db\deployment.json  ✅
  fingerprints.json (optionnel)       -antipiracy\backend\db\fingerprints.json  ✅

── Services ────────────────────────────────────────────────
  IPFS daemon                         http://127.0.0.1:5001  ✅
  Ganache                             http://127.0.0.1:7545  ✅

  ──────────────────────────────────────────────────
  Prérequis critiques : ✅ TOUS VÉRIFIÉS

══════════════════════════════════════════════════════════════
  INITIALISATION DES COMPOSANTS
══════════════════════════════════════════════════════════════

── FingerprintEngine (GraFPrint + Librosa) ─────────────────
  Mode                                GraFPrint GNN  ✅
  Device                              cpu  ℹ️ 
  Temps charg

In [2]:
# %% [markdown]
# ---
# ## 2. Corpus de test (10 s, extraction directe)

# %%
header("CORPUS DE TEST (10 secondes, extraction directe)", BLUE)

DATA_DIR = Path("../data")

# ── Chargement direct (sans conversion bytes) ──────────────────────────
def load_paths(files):
    return [str(f) for f in sorted(files)]

ref_paths    = load_paths((DATA_DIR / "references").glob("*.wav"))
pirate_paths = load_paths((DATA_DIR / "pirates").glob("*.wav"))

# ── Génération de légaux synthétiques très discriminants ───────────────
def make_chirp(duration=10.0, sr=8000):
    t = np.linspace(0, duration, int(sr*duration), endpoint=False)
    # balayage de 100 Hz à 4000 Hz (Nyquist)
    f0, f1 = 100.0, 4000.0
    k = (f1 - f0) / duration
    phase = 2 * np.pi * (f0 * t + 0.5 * k * t**2)
    sig = np.sin(phase).astype(np.float32)
    return sig / (np.abs(sig).max() + 1e-8)

def make_click_train(duration=10.0, sr=8000, click_rate=10):
    sig = np.zeros(int(sr*duration), dtype=np.float32)
    for i in range(0, len(sig), sr//click_rate):
        sig[i] = 1.0
    return sig

synth_dir = Path(tempfile.gettempdir()) / "music_antipiracy_synth"
synth_dir.mkdir(exist_ok=True)

chirp_path = str(synth_dir / "chirp.wav")
click_path = str(synth_dir / "click_train.wav")
sf.write(chirp_path, make_chirp(), 8000)
sf.write(click_path, make_click_train(), 8000)

legal_paths = [chirp_path, click_path]

# ── Définition des objets OEUVRES, PIRATES, LEGAUX ────────────────────
OEUVRES = []
for i, p in enumerate(ref_paths):
    genre = Path(p).stem.split(".")[0].capitalize()
    OEUVRES.append({
        "title":      f"{genre} Original",
        "artist":     f"Artiste {genre}",
        "path":       p,
        "recipients": [ARTIST, PRODUCER],
        "shares":     [70, 30],
    })
    result_row(f"Œuvre {i+1}", f"{OEUVRES[-1]['title']:25s}", OK)

PIRATES = []
for i, p in enumerate(pirate_paths[:len(OEUVRES)]):
    genre = Path(p).stem.replace("pirate_", "").split(".")[0].capitalize()
    source_idx = next((j for j, o in enumerate(OEUVRES) if genre.lower() in o['title'].lower()), i)
    PIRATES.append({
        "label":      f"Copie {genre}",
        "source_idx": source_idx,
        "path":       p,
    })
    result_row(f"Copie {i+1}", f"{PIRATES[-1]['label']:25s}", WARN)

LEGAUX = []
for i, p in enumerate(legal_paths[:len(OEUVRES)]):
    label = Path(p).stem
    LEGAUX.append({
        "label": f"Légal {label}",
        "path":  p,
    })
    result_row(f"Légal {i+1}", f"{LEGAUX[-1]['label']:25s}", INFO)

print(f"\n  Corpus : {len(OEUVRES)} refs, {len(PIRATES)} pirates, {len(LEGAUX)} légaux")


══════════════════════════════════════════════════════════════
  CORPUS DE TEST (10 secondes, extraction directe)
══════════════════════════════════════════════════════════════
  Œuvre 1                             Blues Original             ✅
  Œuvre 2                             Classical Original         ✅
  Œuvre 3                             Country Original           ✅
  Œuvre 4                             Disco Original             ✅
  Œuvre 5                             Hiphop Original            ✅
  Œuvre 6                             Jazz Original              ✅
  Œuvre 7                             Metal Original             ✅
  Œuvre 8                             Pop Original               ✅
  Œuvre 9                             Reggae Original            ✅
  Œuvre 10                            Rock Original              ✅
  Copie 1                             Copie Blues                ⚠️ 
  Copie 2                             Copie Classical            ⚠️ 
  Copie 3     

In [3]:
# %% [markdown]
# ---
# ## 3. FLUX 1 — Étape 1 : Enregistrement des œuvres

# %%
header("FLUX 1 — ÉTAPE 1 : ENREGISTREMENT DES ŒUVRES", "\033[91m")

print(f"""
  Pipeline :
  Fichier audio
    → FingerprintEngine.extract_fingerprint_from_bytes()   [GraFPrint/Librosa]
    → IPFSStorage.upload_file()                             [IPFS]
    → BlockchainManager.register_work()                    [MusicRegistry]
    → FingerprintIndex.add()                               [Index local]
""")

registration_results = []
timings_register     = []

for i, oeuvre in enumerate(OEUVRES):
    section(f"Enregistrement Œuvre {i+1} — {oeuvre['title']}")
    t_start = time.perf_counter()
    steps   = {}

    # ── 1a. Extraction de l'empreinte ─────────────────────────────────────
    t0             = time.perf_counter()
    embedding = engine.extract_fingerprint(oeuvre["path"])
    work_hash      = FingerprintEngine.embedding_to_hash(embedding)
    steps["extract"] = (time.perf_counter() - t0) * 1000

    result_row("Empreinte extraite",
               f"shape={embedding.shape} | hash={work_hash[:20]}...",
               OK)

    # ── Vérification doublon ───────────────────────────────────────────────
    if index.exists(work_hash):
        result_row("Statut", "Déjà enregistrée — mise à jour ignorée", WARN)
        oeuvre["work_hash"] = work_hash
        existing = index.get(work_hash)
        oeuvre["ipfs_cid"]  = existing["ipfs_cid"]
        oeuvre["tx_hash"]   = existing["tx_hash"]
        registration_results.append({**oeuvre, "work_hash": work_hash, "status": "existing"})
        continue

    # ── 1b. Upload IPFS ────────────────────────────────────────────────────
    t0             = time.perf_counter()
    # Lire le fichier depuis le disque
    with open(oeuvre["path"], "rb") as f:
        audio_bytes = f.read()
    t0 = time.perf_counter()
    ipfs_cid = ipfs.upload_file(audio_bytes, filename=f"oeuvre_{i+1}.wav")
    steps["ipfs"]  = (time.perf_counter() - t0) * 1000

    result_row("CID IPFS", ipfs_cid, OK)

    # ── 1c. Enregistrement on-chain ────────────────────────────────────────
    t0              = time.perf_counter()
    tx_hash         = blockchain.register_work(
        work_hash  = work_hash,
        cid        = ipfs_cid,
        recipients = oeuvre["recipients"],
        shares     = oeuvre["shares"]
    )
    steps["chain"]  = (time.perf_counter() - t0) * 1000

    receipt     = w3.eth.get_transaction_receipt(tx_hash)
    gas_used    = receipt["gasUsed"]

    result_row("Transaction",  f"{tx_hash[:20]}...", OK)
    result_row("Gas utilisé",  f"{gas_used:,}", INFO)

    # ── 1d. Vérification on-chain ──────────────────────────────────────────
    is_registered = blockchain.verify_registration(work_hash)
    result_row("Vérification on-chain",
               "isWorkRegistered() → True" if is_registered else "ÉCHEC",
               OK if is_registered else FAIL)

    # ── 1e. Indexation locale ──────────────────────────────────────────────
    t0               = time.perf_counter()
    index.add(
        embedding  = embedding,
        title      = oeuvre["title"],
        artist     = oeuvre["artist"],
        ipfs_cid   = ipfs_cid,
        tx_hash    = tx_hash,
        recipients = [r for r in oeuvre["recipients"]],
        shares     = oeuvre["shares"]
    )
    steps["index"] = (time.perf_counter() - t0) * 1000

    t_total = (time.perf_counter() - t_start) * 1000
    timings_register.append(t_total)

    # Sauvegarde pour les étapes suivantes
    oeuvre["work_hash"] = work_hash
    oeuvre["ipfs_cid"]  = ipfs_cid
    oeuvre["tx_hash"]   = tx_hash
    oeuvre["gas_used"]  = gas_used
    oeuvre["steps"]     = steps

    registration_results.append({**oeuvre, "status": "new"})

    result_row("─" * 30, "", "")
    result_row("Décomposition temps",
               f"extract={steps['extract']:.0f}ms | "
               f"IPFS={steps['ipfs']:.0f}ms | "
               f"chain={steps['chain']:.0f}ms", INFO)
    result_row("⏱  Temps total étape 1",
               f"{t_total:.0f}ms", OK)

# ── Résumé enregistrement ──────────────────────────────────────────────────────
section("Résumé — Étape 1")
new_count = sum(1 for r in registration_results if r["status"] == "new")
result_row("Œuvres nouvellement enregistrées", str(new_count), OK)
result_row("Œuvres déjà présentes",
           str(len(registration_results) - new_count), INFO)
result_row("Index local",
           f"{index.count()} empreintes", OK)
if timings_register:
    result_row("Latence moyenne enregistrement",
               f"{np.mean(timings_register):.0f}ms", INFO)

print(f"\n{GREEN}{BOLD}  ✅ Étape 1 — Enregistrement validé{RESET}")


══════════════════════════════════════════════════════════════
  FLUX 1 — ÉTAPE 1 : ENREGISTREMENT DES ŒUVRES
══════════════════════════════════════════════════════════════

  Pipeline :
  Fichier audio
    → FingerprintEngine.extract_fingerprint_from_bytes()   [GraFPrint/Librosa]
    → IPFSStorage.upload_file()                             [IPFS]
    → BlockchainManager.register_work()                    [MusicRegistry]
    → FingerprintIndex.add()                               [Index local]


── Enregistrement Œuvre 1 — Blues Original ─────────────────
  Empreinte extraite                  shape=(128,) | hash=256fe2ce52e2ab6062c5...  ✅
  CID IPFS                            QmfZJRu5HhWs9r3V9bYAMAUKtaj9w1Vw37ZbtsB34rkGrt  ✅
  Transaction                         0x151d977902fbd2d626...  ✅
  Gas utilisé                         277,537  ℹ️ 
  Vérification on-chain               isWorkRegistered() → True  ✅
  ──────────────────────────────        
  Décomposition temps                 extr

In [4]:
# %% [markdown]
# ---
# ## 4. FLUX 1 — Étape 2 : Détection des copies pirates

# %%
header("FLUX 1 — ÉTAPE 2 : DÉTECTION DES COPIES PIRATES", "\033[91m")

print(f"""
  Pipeline :
  Fichier suspect
    → FingerprintEngine.extract_fingerprint_from_bytes()
    → FingerprintIndex.search()                [similarité cosinus]
    → Décision : score ≥ {detector._threshold} → PIRATERIE | score < {detector._threshold} → LÉGAL
""")

detection_results = []
timings_detect    = []

# ── Test sur les copies pirates ────────────────────────────────────────────────
section("Test A — Copies pirates (résultat attendu : PIRATERIE DÉTECTÉE)")

for i, pirate in enumerate(PIRATES):
    t0     = time.perf_counter()
    # Lecture des bytes si pas déjà présent
    if "bytes" not in pirate:
        with open(pirate["path"], "rb") as f:
            pirate["bytes"] = f.read()
    match  = detector.analyze_content(pirate["bytes"])
    t_ms   = (time.perf_counter() - t0) * 1000

    expected_title = OEUVRES[pirate["source_idx"]]["title"]
    correct        = match.is_match

    result_row(
        f"Copie {i+1} — {pirate['label']}",
        f"score={match.score:.4f} | "
        f"match={'OUI' if match.is_match else 'NON'} | "
        f"{t_ms:.0f}ms",
        OK if correct else FAIL
    )

    if match.is_match:
        print(f"     {INFO} Œuvre identifiée : {match.original_title}")
        print(f"     {INFO} Confiance        : {match.confidence_label}")
    else:
        print(f"     {FAIL} FAUX NÉGATIF — copie pirate non détectée !")

    detection_results.append({
        "type":     "pirate",
        "label":    pirate["label"],
        "is_match": match.is_match,
        "score":    match.score,
        "correct":  correct,
        "latency":  t_ms,
        "match":    match,
        "bytes": open(pirate["path"], "rb").read()
    })

# ── Test sur les fichiers légaux ───────────────────────────────────────────────
section("Test B — Fichiers légaux (résultat attendu : AUCUNE CORRESPONDANCE)")

for i, legal in enumerate(LEGAUX):
    t0     = time.perf_counter()
    # Lecture des bytes si pas déjà présent
    if "bytes" not in legal:
        with open(legal["path"], "rb") as f:
            legal["bytes"] = f.read()
    match  = detector.analyze_content(legal["bytes"])
    t_ms   = (time.perf_counter() - t0) * 1000

    correct = not match.is_match   # on attend is_match = False

    result_row(
        f"Légal {i+1} — {legal['label']}",
        f"score={match.score:.4f} | "
        f"match={'OUI' if match.is_match else 'NON'} | "
        f"{t_ms:.0f}ms",
        OK if correct else FAIL
    )

    if match.is_match:
        print(f"     {FAIL} FAUX POSITIF — fichier légitime incorrectement signalé !")

    detection_results.append({
        "type":    "legal",
        "label":   legal["label"],
        "is_match": match.is_match,
        "score":   match.score,
        "correct": correct,
        "latency": t_ms,
        "match":   match,
        "bytes": open(legal["path"], "rb").read()
    })

# ── Métriques de détection ────────────────────────────────────────────────────
section("Métriques de détection (tableau 4.1)")

pirate_results = [r for r in detection_results if r["type"] == "pirate"]
legal_results  = [r for r in detection_results if r["type"] == "legal"]

tp = sum(1 for r in pirate_results if r["is_match"])
fn = sum(1 for r in pirate_results if not r["is_match"])
fp = sum(1 for r in legal_results  if r["is_match"])
tn = sum(1 for r in legal_results  if not r["is_match"])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

pirate_scores = [r["score"] for r in pirate_results]
legal_scores  = [r["score"] for r in legal_results]

print(f"""
  Matrice de confusion
  ┌─────────────────────────────────────────────────┐
  │  Vrais Positifs  (TP) : {tp:2d}  │  Faux Positifs  (FP) : {fp:2d}  │
  │  Faux Négatifs   (FN) : {fn:2d}  │  Vrais Négatifs (TN) : {tn:2d}  │
  └─────────────────────────────────────────────────┘

  Précision  : {precision:.4f}  ({precision*100:.1f}%)
  Rappel     : {recall:.4f}  ({recall*100:.1f}%)
  F1-score   : {f1:.4f}

  Score moyen — copies pirates : {np.mean(pirate_scores):.4f}
  Score moyen — fichiers légaux : {np.mean(legal_scores):.4f}
  Séparabilité (Δ) : {np.mean(pirate_scores) - np.mean(legal_scores):.4f}

  Seuil utilisé : {detector._threshold}
  Latence moyenne détection : {np.mean(timings_detect):.1f}ms
""")

all_correct = all(r["correct"] for r in detection_results)
print(f"  {OK if all_correct else FAIL} "
      f"{'Toutes les décisions correctes' if all_correct else 'Erreurs de classification'}")

print(f"\n{GREEN}{BOLD}  ✅ Étape 2 — Détection validée{RESET}")


══════════════════════════════════════════════════════════════
  FLUX 1 — ÉTAPE 2 : DÉTECTION DES COPIES PIRATES
══════════════════════════════════════════════════════════════

  Pipeline :
  Fichier suspect
    → FingerprintEngine.extract_fingerprint_from_bytes()
    → FingerprintIndex.search()                [similarité cosinus]
    → Décision : score ≥ 0.85 → PIRATERIE | score < 0.85 → LÉGAL


── Test A — Copies pirates (résultat attendu : PIRATERIE DÉTECTÉE) 
  Copie 1 — Copie Blues               score=0.9282 | match=OUI | 23139ms  ✅
     ℹ️  Œuvre identifiée : Blues Original
     ℹ️  Confiance        : HAUTE
  Copie 2 — Copie Classical           score=0.9304 | match=OUI | 23420ms  ✅
     ℹ️  Œuvre identifiée : Classical Original
     ℹ️  Confiance        : HAUTE
  Copie 3 — Copie Country             score=0.8541 | match=OUI | 23030ms  ✅
     ℹ️  Œuvre identifiée : Country Original
     ℹ️  Confiance        : HAUTE
  Copie 4 — Copie Disco               score=0.9449 | match=OUI | 2

In [5]:
# %% [markdown]
# ---
# ## 5. FLUX 1 — Étape 3 : Génération des preuves certifiées

# %%
header("FLUX 1 — ÉTAPE 3 : GÉNÉRATION DES PREUVES CERTIFIÉES", "\033[91m")

print(f"""
  Pipeline :
  MatchResult (is_match=True)
    → IPFSStorage.upload_file(audio_suspect)          → suspect_cid
    → IPFSStorage.upload_json(rapport_forensique)     → report_cid
    → BlockchainManager.store_proof(report_cid, hash) → evidence_hash + tx_hash
    → Proof certifiée
""")

proof_results  = []
timings_proof  = []
positive_matches = [r for r in detection_results
                    if r["type"] == "pirate" and r["is_match"]]

for i, det_result in enumerate(positive_matches):
    section(f"Génération preuve {i+1}/{len(positive_matches)} — {det_result['label']}")
    t0 = time.perf_counter()

    # ── Génération de la preuve ────────────────────────────────────────────
    proof = detector.generate_proof(
        match        = det_result["match"],
        suspect_audio_bytes = det_result["bytes"]
    )
    t_ms  = (time.perf_counter() - t0) * 1000
    timings_proof.append(t_ms)

    result_row("ID preuve",            proof.id[:20] + "...", OK)
    result_row("CID suspect (IPFS)",   proof.suspect_cid, OK)
    result_row("CID rapport (IPFS)",   proof.report_cid, OK)
    result_row("Evidence hash",
               (proof.evidence_hash or "")[:20] + "...", OK)
    result_row("Transaction blockchain",
               (proof.tx_hash or "")[:20] + "...", OK)
    result_row("Score de match",       f"{proof.match_score:.4f}", INFO)

    # ── Vérification IPFS ─────────────────────────────────────────────────
    suspect_ok = ipfs.verify_integrity(proof.suspect_cid)
    report_ok  = ipfs.verify_integrity(proof.report_cid)
    result_row("Intégrité suspect IPFS",
               "Vérifiée" if suspect_ok else "ÉCHEC",
               OK if suspect_ok else FAIL)
    result_row("Intégrité rapport IPFS",
               "Vérifiée" if report_ok else "ÉCHEC",
               OK if report_ok else FAIL)

    # ── Vérification contenu du rapport ───────────────────────────────────
    report_bytes = ipfs.retrieve_file(proof.report_cid)
    report_data  = json.loads(report_bytes.decode("utf-8"))
    report_ok_content = (
        report_data.get("match_score") == proof.match_score and
        report_data.get("suspect_cid") == proof.suspect_cid
    )
    result_row("Cohérence rapport JSON",
               "Vérifiée" if report_ok_content else "INCOHÉRENTE",
               OK if report_ok_content else FAIL)

    # ── Gas de la certification ───────────────────────────────────────────
    if proof.tx_hash:
        receipt  = w3.eth.get_transaction_receipt(proof.tx_hash)
        gas_cert = receipt["gasUsed"]
        result_row("Gas certifyInfringement", f"{gas_cert:,}", INFO)
    else:
        gas_cert = 0

    result_row("⏱  Temps total étape 3", f"{t_ms:.0f}ms", INFO)

    # Affichage du rapport forensique
    print(f"\n  {BOLD}Rapport forensique :{RESET}")
    print("  " + "\n  ".join(proof.generate_dmca_rapport().split("\n")))

    proof_results.append({
        "proof":     proof,
        "t_ms":      t_ms,
        "gas_cert":  gas_cert,
        "ipfs_ok":   suspect_ok and report_ok,
        "content_ok": report_ok_content
    })

# ── Résumé étape 3 ────────────────────────────────────────────────────────────
section("Résumé — Étape 3")
result_row("Preuves générées",         str(len(proof_results)), OK)
result_row("Intégrité IPFS",
           "100%" if all(p["ipfs_ok"] for p in proof_results) else "PARTIELLE",
           OK if all(p["ipfs_ok"] for p in proof_results) else WARN)
result_row("Cohérence contenu",
           "100%" if all(p["content_ok"] for p in proof_results) else "PARTIELLE",
           OK if all(p["content_ok"] for p in proof_results) else WARN)
if timings_proof:
    result_row("Latence moyenne étape 3",
               f"{np.mean(timings_proof):.0f}ms", INFO)
if proof_results:
    result_row("Gas moyen certification",
               f"{np.mean([p['gas_cert'] for p in proof_results]):.0f}", INFO)

print(f"\n{GREEN}{BOLD}  ✅ Étape 3 — Certification validée{RESET}")


══════════════════════════════════════════════════════════════
  FLUX 1 — ÉTAPE 3 : GÉNÉRATION DES PREUVES CERTIFIÉES
══════════════════════════════════════════════════════════════

  Pipeline :
  MatchResult (is_match=True)
    → IPFSStorage.upload_file(audio_suspect)          → suspect_cid
    → IPFSStorage.upload_json(rapport_forensique)     → report_cid
    → BlockchainManager.store_proof(report_cid, hash) → evidence_hash + tx_hash
    → Proof certifiée


── Génération preuve 1/10 — Copie Blues ────────────────────
  ID preuve                           77cb0785-6b9a-416b-8...  ✅
  CID suspect (IPFS)                  QmQTeHiNYpSH6TSvo8JfQ5wjfFBWHkxrF4dHYLFbdmU7Bq  ✅
  CID rapport (IPFS)                  QmQY7kqYjuXf1agPZLgSA3ai7fCow9Aaag1NGYWHn5aFdq  ✅
  Evidence hash                       251298908da505236583...  ✅
  Transaction blockchain              0xe5840e4d44faaba257...  ✅
  Score de match                      0.9282  ℹ️ 
  Intégrité suspect IPFS              Vérifiée  ✅
  I

In [6]:
# %% [markdown]
# ---
# ## 6. FLUX 1 — Étape 4 : Simulation de la notification DMCA

# %%
header("FLUX 1 — ÉTAPE 4 (SIMULÉE) : NOTIFICATION DMCA", "\033[93m")

print(f"""
  Statut : SIMULÉE via webhook HTTP

  Dans un déploiement en production, ce webhook serait remplacé par :
    - L'API YouTube Content ID
    - L'API Spotify for Artists
    - L'API TikTok Creator Marketplace

  Lacune centrale documentée par Shi & Zhou (2025) et Mareckova (2024) :
  aucun système académique ne propose de mécanisme opérationnel permettant
  de déclencher automatiquement un retrait depuis un événement smart contract.

  Ce prototype fournit la couche de déclenchement simulée.
""")

takedown_results = []
WEBHOOK_URL = "https://webhook.site/cb8467a8-9d6a-46cb-be8e-94a1ca05f44c"  # ← Remplacer par votre URL

for i, pr in enumerate(proof_results):
    proof = pr["proof"]
    section(f"Notification DMCA {i+1} — Preuve {proof.id[:16]}...")

    takedown_id = str(uuid.uuid4())

    # Payload structuré conforme à une notification DMCA standard
    payload = {
        "takedown_id":      takedown_id,
        "generated_at":     datetime.utcnow().isoformat(),
        "evidence_cid":     proof.report_cid,
        "evidence_hash":    proof.evidence_hash,
        "blockchain_tx":    proof.tx_hash,
        "match_score":      proof.match_score,
        "original_work":    proof.original_work_hash,
        "certification": {
            "blockchain":   settings.ganache_url,
            "contract":     dep["address"],
            "ipfs_gateway": ipfs.get_gateway_url(proof.report_cid),
        },
        "dmca_notice": (
            "This content has been identified as an unauthorized copy "
            "of a registered work. Proof of infringement has been "
            "certified on the blockchain. Please remove this content "
            "immediately per DMCA Section 512."
        )
    }

    result_row("Takedown ID", takedown_id[:20] + "...", INFO)
    result_row("Payload généré", f"{len(json.dumps(payload))} bytes", OK)

    # Tentative d'envoi webhook réel
    t0       = time.perf_counter()
    status   = TakedownStatus.SIMULATED
    response = None

    try:
        r = _req.post(WEBHOOK_URL, json=payload, timeout=5)
        response = {"status_code": r.status_code, "body": r.text[:100]}
        status   = TakedownStatus.SENT
        result_row("Webhook",
                   f"HTTP {r.status_code}",
                   OK if r.status_code == 200 else WARN)
    except Exception as e:
        # Simulation locale si webhook inaccessible
        response = {
            "simulated":    True,
            "payload_size": len(json.dumps(payload)),
            "note":         "Webhook non configuré — simulation locale"
        }
        result_row("Webhook",
                   "Non configuré — simulation locale",
                   WARN)

    t_ms = (time.perf_counter() - t0) * 1000

    takedown = TakedownRequest(
        id               = takedown_id,
        infringing_url   = "https://plateforme-tierce.example.com/contenu-pirate",
        proof_hash       = proof.evidence_hash or "",
        proof_cid        = proof.report_cid,
        tx_hash          = proof.tx_hash,
        status           = status,
        webhook_response = response
    )

    result_row("Statut notification", takedown.status.value,
               OK if status == TakedownStatus.SENT else WARN)
    result_row("Temps génération",   f"{t_ms:.1f}ms", INFO)

    # Notice DMCA textuelle
    print(f"\n  {BOLD}Notice DMCA générée :{RESET}")
    print("  " + "\n  ".join(takedown.generate_dmca_notice().split("\n")))

    takedown_results.append({
        "takedown": takedown,
        "payload":  payload,
        "t_ms":     t_ms
    })

# ── Résumé étape 4 ────────────────────────────────────────────────────────────
section("Résumé — Étape 4 (simulée)")
result_row("Notifications générées",    str(len(takedown_results)), OK)
result_row("Statut",
           "SENT" if all(t["takedown"].status == TakedownStatus.SENT
                         for t in takedown_results)
           else "SIMULATED", WARN)
result_row("Lacune documentée",
           "Étape non automatisée sur plateformes tierces", WARN)

print(f"\n{YELLOW}{BOLD}  ⚠️  Étape 4 — Simulée (webhook local) — Lacune centrale du champ{RESET}")


══════════════════════════════════════════════════════════════
  FLUX 1 — ÉTAPE 4 (SIMULÉE) : NOTIFICATION DMCA
══════════════════════════════════════════════════════════════

  Statut : SIMULÉE via webhook HTTP

  Dans un déploiement en production, ce webhook serait remplacé par :
    - L'API YouTube Content ID
    - L'API Spotify for Artists
    - L'API TikTok Creator Marketplace

  Lacune centrale documentée par Shi & Zhou (2025) et Mareckova (2024) :
  aucun système académique ne propose de mécanisme opérationnel permettant
  de déclencher automatiquement un retrait depuis un événement smart contract.

  Ce prototype fournit la couche de déclenchement simulée.


── Notification DMCA 1 — Preuve 77cb0785-6b9a-41... ────────
  Takedown ID                         a74c2071-888d-4882-9...  ℹ️ 
  Payload généré                      886 bytes  ✅
  Webhook                             HTTP 429  ⚠️ 
  Statut notification                 SENT  ✅
  Temps génération                    937.8ms  

In [7]:
# %% [markdown]
# ---
# ## 7. FLUX 2 — Étape 5 : Simulation de distribution des redevances

# %%
header("FLUX 2 — ÉTAPE 5 : SIMULATION DE DISTRIBUTION DES REDEVANCES", "\033[92m")

print(f"""
  Pipeline :
  Artiste sélectionne une œuvre + saisit un montant fictif
    → BlockchainManager.simulate_royalty_payment(work_hash, totalAmount)
    → MusicRegistry.simulateRoyaltyPayment()          [Solidity on-chain]
    → loop : montant_i = (totalAmount × share_i) / 100
    → emit PaymentSimulated(beneficiary, amount, timestamp)
    → Événements capturés par Web3.py
    → Affichage tableau de bord

  Fondement : Ciriello et al. (2023) — paiements automatiques via smart contracts
  Validation : Pan et al. (2024) — équité par modélisation par agents
""")

royalty_results  = []
timings_royalty  = []
MONTANTS_TEST    = [1000, 500, 2500]

for i, (oeuvre, montant) in enumerate(zip(OEUVRES, MONTANTS_TEST)):
    if "work_hash" not in oeuvre:
        print(f"  {WARN} Œuvre {i+1} sans work_hash — enregistrement nécessaire")
        continue

    section(f"Simulation {i+1} — {oeuvre['title']} | Montant : {montant} unités")

    result_row("Œuvre",          oeuvre["title"], INFO)
    result_row("Hash empreinte", oeuvre["work_hash"][:20] + "...", INFO)
    result_row("Bénéficiaires",  str(len(oeuvre["recipients"])), INFO)
    result_row("Parts",
               " + ".join(f"{s}%" for s in oeuvre["shares"]),
               INFO)
    result_row("Montant fictif", f"{montant} unités", INFO)

    # ── Appel simulateRoyaltyPayment ───────────────────────────────────────
    t0                   = time.perf_counter()
    tx_hash, payments    = blockchain.simulate_royalty_payment(
        work_hash    = oeuvre["work_hash"],
        total_amount = montant
    )
    t_ms = (time.perf_counter() - t0) * 1000
    timings_royalty.append(t_ms)

    result_row("Transaction", f"{tx_hash[:20]}...", OK)
    result_row("Temps", f"{t_ms:.0f}ms", INFO)

    # ── Vérification des paiements ────────────────────────────────────────
    total_distributed = sum(p["amount"] for p in payments)
    total_ok          = total_distributed == montant
    count_ok          = len(payments) == len(oeuvre["recipients"])

    result_row("Événements PaymentSimulated",
               f"{len(payments)} / {len(oeuvre['recipients'])} attendus",
               OK if count_ok else FAIL)
    result_row("Total distribué",
               f"{total_distributed} / {montant}",
               OK if total_ok else FAIL)

    # ── Détail de la répartition ──────────────────────────────────────────
    print(f"\n  Répartition détaillée :")
    print(f"  {'Bénéficiaire':42s} {'Part':6s} {'Montant':10s} {'Vérification':12s}")
    print("  " + "─" * 72)

    for j, (p, share) in enumerate(zip(payments, oeuvre["shares"])):
        expected   = (montant * share) // 100
        is_correct = p["amount"] == expected
        pct_actual = p["amount"] / montant * 100

        print(f"  {p['beneficiary'][:38]:40s} "
              f"{share:4d}%  "
              f"{p['amount']:8d}  "
              f"{'✅ correct' if is_correct else '❌ '+ str(expected)}")

    # ── Gas de la simulation ──────────────────────────────────────────────
    receipt   = w3.eth.get_transaction_receipt(tx_hash)
    gas_used  = receipt["gasUsed"]
    gas_per_b = gas_used // len(payments) if payments else 0

    result_row("Gas total simulation",
               f"{gas_used:,}", INFO)
    result_row("Gas par bénéficiaire (estimé)",
               f"~{gas_per_b:,}", INFO)

    royalty_results.append({
        "oeuvre":             oeuvre["title"],
        "work_hash":          oeuvre["work_hash"],
        "montant":            montant,
        "payments":           payments,
        "total_distributed":  total_distributed,
        "total_ok":           total_ok,
        "count_ok":           count_ok,
        "tx_hash":            tx_hash,
        "gas_used":           gas_used,
        "t_ms":               t_ms
    })

# ── Résumé étape 5 ────────────────────────────────────────────────────────────
section("Résumé — Étape 5")
all_ok_royalty = all(r["total_ok"] and r["count_ok"] for r in royalty_results)

result_row("Simulations exécutées",     str(len(royalty_results)), OK)
result_row("Exactitude distributions",
           "100%" if all_ok_royalty else "PARTIELLE",
           OK if all_ok_royalty else FAIL)
if timings_royalty:
    result_row("Latence moyenne étape 5",
               f"{np.mean(timings_royalty):.0f}ms", INFO)
if royalty_results:
    result_row("Gas moyen simulation",
               f"{np.mean([r['gas_used'] for r in royalty_results]):.0f}", INFO)

# Vérification événements on-chain
contract = w3.eth.contract(
    address = Web3.to_checksum_address(dep["address"]),
    abi     = dep["abi"]
)
pay_filter = contract.events.PaymentSimulated.create_filter(from_block=0)
pay_events = pay_filter.get_all_entries()
result_row("Événements PaymentSimulated on-chain",
           str(len(pay_events)), OK)

print(f"\n{GREEN}{BOLD}  ✅ Étape 5 — Simulation de redevances validée{RESET}")


══════════════════════════════════════════════════════════════
  FLUX 2 — ÉTAPE 5 : SIMULATION DE DISTRIBUTION DES REDEVANCES
══════════════════════════════════════════════════════════════

  Pipeline :
  Artiste sélectionne une œuvre + saisit un montant fictif
    → BlockchainManager.simulate_royalty_payment(work_hash, totalAmount)
    → MusicRegistry.simulateRoyaltyPayment()          [Solidity on-chain]
    → loop : montant_i = (totalAmount × share_i) / 100
    → emit PaymentSimulated(beneficiary, amount, timestamp)
    → Événements capturés par Web3.py
    → Affichage tableau de bord

  Fondement : Ciriello et al. (2023) — paiements automatiques via smart contracts
  Validation : Pan et al. (2024) — équité par modélisation par agents


── Simulation 1 — Blues Original | Montant : 1000 unités ───
  Œuvre                               Blues Original  ℹ️ 
  Hash empreinte                      256fe2ce52e2ab6062c5...  ℹ️ 
  Bénéficiaires                       2  ℹ️ 
  Parts            

In [8]:
# %% [markdown]
# ---
# ## 8. Mesures de performance globales

# %%
header("MESURES DE PERFORMANCE GLOBALES", BLUE)

# ─────────────────────────────────────────────────────────────────────────────
# Récapitulatif des latences par étape
# ─────────────────────────────────────────────────────────────────────────────
section("Latences par étape (tableau 4.3)")

print(f"""
  {'Étape':40s} {'Moy (ms)':12s} {'Min (ms)':12s} {'Max (ms)':12s}
  {"─" * 78}""")

latency_data = {}

if timings_register:
    latency_data["Étape 1 — Enregistrement"]  = timings_register
if timings_detect:
    latency_data["Étape 2 — Détection"]        = timings_detect
if timings_proof:
    latency_data["Étape 3 — Certification"]    = timings_proof
if timings_royalty:
    latency_data["Étape 5 — Simulation royalty"] = timings_royalty

for label, times in latency_data.items():
    arr = np.array(times)
    print(f"  {label:40s} "
          f"{np.mean(arr):10.1f}   "
          f"{np.min(arr):10.1f}   "
          f"{np.max(arr):10.1f}")

# Pipeline complet (étapes 2+3)
if timings_detect and timings_proof:
    pipeline_total = [d + p for d, p in zip(
        timings_detect[:len(proof_results)],
        timings_proof
    )]
    arr = np.array(pipeline_total)
    print(f"  {'Pipeline complet (étapes 2+3)':40s} "
          f"{np.mean(arr):10.1f}   "
          f"{np.min(arr):10.1f}   "
          f"{np.max(arr):10.1f}")
    print(f"\n  Référence Chen et al. (2022) : objectif < 1000ms (1 seconde)")
    print(f"  Notre pipeline complet       : {np.mean(arr):.0f}ms "
          f"{'✅ objectif atteint' if np.mean(arr) < 1000 else '⚠️  dépasse 1s'}")

# ─────────────────────────────────────────────────────────────────────────────
# Coûts gas récapitulatifs
# ─────────────────────────────────────────────────────────────────────────────
section("Coûts gas (tableau 4.5)")

gas_register  = [o.get("gas_used", 0) for o in OEUVRES if "gas_used" in o]
gas_certify   = [p["gas_cert"] for p in proof_results if p["gas_cert"] > 0]
gas_royalty   = [r["gas_used"] for r in royalty_results]

GAS_PRICE_GWEI   = 20
MATIC_USD        = 0.85

print(f"""
  {'Fonction':35s} {'Gas utilisé':14s} {'Coût Polygon (~$)':20s}
  {"─" * 72}""")

for label, gas_list in [
    ("registerWork",           gas_register),
    ("certifyInfringement",    gas_certify),
    ("simulateRoyaltyPayment", gas_royalty)
]:
    if gas_list:
        avg_gas  = np.mean(gas_list)
        cost_eth = avg_gas * GAS_PRICE_GWEI / 1e9
        cost_usd = cost_eth * MATIC_USD / 1e9 * 1e9
        print(f"  {label:35s} {avg_gas:12,.0f}   ~${cost_eth * 2000:.4f}")

total_gas = (np.mean(gas_register)  if gas_register  else 0 +
             np.mean(gas_certify)   if gas_certify   else 0 +
             np.mean(gas_royalty)   if gas_royalty   else 0)

if total_gas > 0:
    cost_total = total_gas * GAS_PRICE_GWEI / 1e9 * 2000
    print(f"\n  Pipeline complet (1+3+5) : ~{total_gas:,.0f} gas | ~${cost_total:.4f}")
    print(f"  Environnement Ganache (dev) : coût = 0 (transactions gratuites)")

# ─────────────────────────────────────────────────────────────────────────────
# Événements on-chain récapitulatifs
# ─────────────────────────────────────────────────────────────────────────────
section("Événements on-chain émis")

work_filter = contract.events.WorkRegistered.create_filter(from_block=0)
work_events = work_filter.get_all_entries()
infr_filter = contract.events.InfringementCertified.create_filter(from_block=0)
infr_events = infr_filter.get_all_entries()
pay_filter  = contract.events.PaymentSimulated.create_filter(from_block=0)
pay_events  = pay_filter.get_all_entries()

result_row("WorkRegistered",          str(len(work_events)), OK)
result_row("InfringementCertified",   str(len(infr_events)), OK)
result_row("PaymentSimulated",        str(len(pay_events)), OK)
result_row("Total événements",
           str(len(work_events) + len(infr_events) + len(pay_events)), OK)


══════════════════════════════════════════════════════════════
  MESURES DE PERFORMANCE GLOBALES
══════════════════════════════════════════════════════════════

── Latences par étape (tableau 4.3) ────────────────────────

  Étape                                    Moy (ms)     Min (ms)     Max (ms)    
  ──────────────────────────────────────────────────────────────────────────────
  Étape 1 — Enregistrement                    21263.3      15478.2      25070.1
  Étape 3 — Certification                       622.6        352.2       1182.2
  Étape 5 — Simulation royalty                   61.1         46.0         82.2

── Coûts gas (tableau 4.5) ─────────────────────────────────

  Fonction                            Gas utilisé    Coût Polygon (~$)   
  ────────────────────────────────────────────────────────────────────────
  registerWork                             277,536   ~$11.1014
  certifyInfringement                      140,190   ~$5.6076
  simulateRoyaltyPayment            

In [9]:
# %% [markdown]
# ---
# ## 9. Vérifications d'intégrité du pipeline

# %%
header("VÉRIFICATIONS D'INTÉGRITÉ DU PIPELINE", BLUE)

section("Cohérence IA → IPFS → Blockchain")

integrity_checks = []

for oeuvre in OEUVRES:
    if "work_hash" not in oeuvre:
        continue

    # 1. Hash local = hash on-chain ?
    is_on_chain = blockchain.verify_registration(oeuvre["work_hash"])

    # 2. CID IPFS accessible ?
    ipfs_ok = ipfs.verify_integrity(oeuvre["ipfs_cid"])

    # 3. Données on-chain cohérentes avec l'enregistrement ?
    try:
        chain_data = blockchain.get_work(oeuvre["work_hash"])
        cid_ok     = chain_data["cid"] == oeuvre["ipfs_cid"]
        shares_ok  = chain_data["shares"] == oeuvre["shares"]
    except Exception:
        cid_ok = shares_ok = False

    all_ok = is_on_chain and ipfs_ok and cid_ok and shares_ok
    integrity_checks.append(all_ok)

    print(f"  {oeuvre['title'][:30]:32s} "
          f"on-chain={'✅' if is_on_chain else '❌'} | "
          f"IPFS={'✅' if ipfs_ok else '❌'} | "
          f"CID={'✅' if cid_ok else '❌'} | "
          f"parts={'✅' if shares_ok else '❌'} | "
          f"{'✅ OK' if all_ok else '❌ ERREUR'}")

# Intégrité des preuves
section("Cohérence Preuve → IPFS → Blockchain")

for pr in proof_results:
    proof = pr["proof"]

    # Rapport récupérable
    report_ok = ipfs.verify_integrity(proof.report_cid)
    # Suspect récupérable
    suspect_ok = ipfs.verify_integrity(proof.suspect_cid)
    # Evidence hash non nul
    hash_ok = bool(proof.evidence_hash) and len(proof.evidence_hash) == 64
    # Tx confirmée
    tx_ok = bool(proof.tx_hash)

    all_ok = report_ok and suspect_ok and hash_ok and tx_ok
    integrity_checks.append(all_ok)

    print(f"  Preuve {proof.id[:16]}...  "
          f"rapport={'✅' if report_ok else '❌'} | "
          f"suspect={'✅' if suspect_ok else '❌'} | "
          f"hash={'✅' if hash_ok else '❌'} | "
          f"tx={'✅' if tx_ok else '❌'} | "
          f"{'✅ OK' if all_ok else '❌ ERREUR'}")

all_integrity = all(integrity_checks)
print(f"\n  {'─' * 50}")
print(f"  Intégrité globale : "
      f"{'✅ PIPELINE COHÉRENT' if all_integrity else '❌ INCOHÉRENCES DÉTECTÉES'}")


══════════════════════════════════════════════════════════════
  VÉRIFICATIONS D'INTÉGRITÉ DU PIPELINE
══════════════════════════════════════════════════════════════

── Cohérence IA → IPFS → Blockchain ────────────────────────
  Blues Original                   on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Classical Original               on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Country Original                 on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Disco Original                   on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Hiphop Original                  on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Jazz Original                    on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Metal Original                   on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Pop Original                     on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Reggae Original                  on-chain=✅ | IPFS=✅ | CID=✅ | parts=✅ | ✅ OK
  Rock Original                    on-chain=✅ | IPF

In [10]:
# %% [markdown]
# ---
# ## 10. Tableau de synthèse final

# %%
header("TABLEAU DE SYNTHÈSE FINAL", "\033[95m")

print(f"""
  {'═' * 62}
  RÉSULTATS DU PIPELINE END-TO-END
  Généré le : {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC
  {'═' * 62}
""")

# Flux 1
section("Flux 1 — Piraterie & Enforcement")
result_row("Étape 1 — Enregistrement",
           f"{len([o for o in OEUVRES if 'work_hash' in o])} œuvres",
           OK)
result_row("Étape 2 — Détection (Vrais Positifs)",
           f"{tp}/{tp+fn} ({tp/(tp+fn)*100:.0f}%)",
           OK if fn == 0 else WARN)
result_row("Étape 2 — Faux Positifs",
           f"{fp}/{fp+tn}",
           OK if fp == 0 else FAIL)
result_row("Étape 2 — Précision / Rappel / F1",
           f"{precision:.3f} / {recall:.3f} / {f1:.3f}",
           OK if f1 >= 0.85 else WARN)
result_row("Étape 3 — Preuves certifiées",
           f"{len(proof_results)} preuves",
           OK)
result_row("Étape 3 — Intégrité IPFS",
           "100%" if all(p["ipfs_ok"] for p in proof_results) else "PARTIELLE",
           OK if all(p["ipfs_ok"] for p in proof_results) else WARN)
result_row("Étape 4 — Notifications DMCA",
           f"{len(takedown_results)} (simulées)",
           WARN)

print()
section("Flux 2 — Streaming légal & Redevances")
result_row("Étape 5 — Simulations exécutées",
           str(len(royalty_results)),
           OK)
result_row("Étape 5 — Exactitude distributions",
           "100%" if all_ok_royalty else "PARTIELLE",
           OK if all_ok_royalty else FAIL)
result_row("Étape 5 — Événements PaymentSimulated",
           str(len(pay_events)),
           OK)

print()
section("Performances")
if timings_detect and timings_proof:
    result_row("Latence pipeline complet (moy)",
               f"{np.mean(arr):.0f}ms",
               OK if np.mean(arr) < 1000 else WARN)
result_row("Latence détection seule (moy)",
           f"{np.mean(timings_detect):.0f}ms",
           INFO)
result_row("Latence certification (moy)",
           f"{np.mean(timings_proof):.0f}ms",
           INFO)

print()
section("Intégrité globale")
result_row("Pipeline cohérent",
           "OUI" if all_integrity else "NON",
           OK if all_integrity else FAIL)
result_row("Événements on-chain totaux",
           str(len(work_events) + len(infr_events) + len(pay_events)),
           OK)

print(f"""
  {'─' * 62}
  Étape 4 (Enforcement externe) : SIMULÉE — Lacune centrale du champ
  Cf. Shi & Zhou (2025) ; Mareckova (2024)
  {'─' * 62}

  Conclusion :
  Le pipeline intégré IA+Blockchain est fonctionnel pour les étapes
  opérationnelles (1, 2, 3, 5). Les deux flux sont validés avec des
  performances cohérentes avec les objectifs définis au Chapitre 3.
  {'─' * 62}
""")

# Résultat final
all_passed = (
    all_integrity and
    all_ok_royalty and
    f1 >= 0.80 and
    fp == 0
)

if all_passed:
    print(f"{GREEN}{BOLD}  ✅✅✅  NOTEBOOK 05 VALIDÉ — PIPELINE END-TO-END OPÉRATIONNEL  ✅✅✅{RESET}")
else:
    print(f"{YELLOW}{BOLD}  ⚠️   NOTEBOOK 05 PARTIELLEMENT VALIDÉ — VÉRIFIER CI-DESSUS  ⚠️{RESET}")


══════════════════════════════════════════════════════════════
  TABLEAU DE SYNTHÈSE FINAL
══════════════════════════════════════════════════════════════

  ══════════════════════════════════════════════════════════════
  RÉSULTATS DU PIPELINE END-TO-END
  Généré le : 2026-06-17 07:47:00 UTC
  ══════════════════════════════════════════════════════════════


── Flux 1 — Piraterie & Enforcement ────────────────────────
  Étape 1 — Enregistrement            10 œuvres  ✅
  Étape 2 — Détection (Vrais Positifs) 10/10 (100%)  ✅
  Étape 2 — Faux Positifs             0/2  ✅
  Étape 2 — Précision / Rappel / F1   1.000 / 1.000 / 1.000  ✅
  Étape 3 — Preuves certifiées        10 preuves  ✅
  Étape 3 — Intégrité IPFS            100%  ✅
  Étape 4 — Notifications DMCA        10 (simulées)  ⚠️ 


── Flux 2 — Streaming légal & Redevances ───────────────────
  Étape 5 — Simulations exécutées     3  ✅
  Étape 5 — Exactitude distributions  100%  ✅
  Étape 5 — Événements PaymentSimulated 6  ✅


── Perform

In [11]:
# %% [markdown]
# ---
# ## 11. Nettoyage des fichiers temporaires

# %%
header("NETTOYAGE", BLUE)

# cleaned = 0
# for collection in [OEUVRES, PIRATES, LEGAUX]:
#     for item in collection:
#         path = item.get("path")
#         if path and Path(path).exists():
#             try:
#                 os.unlink(path)
#                 cleaned += 1
#             except Exception:
#                 pass

# print(f"  {cleaned} fichiers temporaires supprimés  {OK}")
# print(f"  Index conservé : {index.count()} empreintes dans {settings.fingerprint_db}")
# print(f"\n  → Résultats disponibles pour le Chapitre 4 (Résultats & Discussion)")
# print(f"  → Captures d'écran à effectuer sur l'interface Vue.js")
# print(f"  → Passer au notebook 06 pour les métriques détaillées")


══════════════════════════════════════════════════════════════
  NETTOYAGE
══════════════════════════════════════════════════════════════
